# BirdCLEF+ 2025 - Pretrained models: A2-to-A3 distillation on cached embeddings

CPU only, no submission. The teacher is the original A2 Perch MLP head (`perch_head.npz`, with its own standardiser) applied to the cached soundscape Perch embeddings. Soft labels: `p ** 1.3`, then `p < 0.01` set to 0.

The student uses the A3 MLP head recipe (hidden 512, dropout 0.2, max_lr 3e-3, wd 1e-2, 30 epochs, batch 1024, one-cycle pct_start 0.1, seed 42, default bias). It trains on the train-fold train-clip Bird-MAE windows with real labels and per-clip weights, plus all soundscape Bird-MAE windows with teacher labels. Each soundscape window has a loss weight of 0.5 (per window), in the same shuffled batches and with the same weighted-mean loss. There are two variants:

**distill:** the source features are not transformed.

**distill_coral:** the train-fold and val source features go through the standardised CORAL map of coral_heads (lambda 1, fitted on the train-fold source and the soundscape target), while the soundscape features are not transformed. The standardiser comes from the transformed train-fold source.

Both heads are saved and there is no selection; validation on the val fold is only recorded. Output: `birdmae_head_distill.npz`, `birdmae_head_distill_coral.npz`, `species.json`, `head_cfg.json`, `distill_results.json`.

In [ ]:
import os
N_THREADS = int(os.environ.get("M3_THREADS", os.cpu_count() or 4))
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[v] = str(N_THREADS)         # set before numpy loads; all cores for the matrix work

import ast, glob, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

T0 = time.time()
torch.set_num_threads(N_THREADS)
SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"     # local test only, never set on Kaggle
OUT = Path(os.environ.get("M3_OUT_DIR", "/kaggle/working"))
print("device: cpu | torch threads:", torch.get_num_threads(), "| os.cpu_count():", os.cpu_count())

CFG = dict(val_frac=0.2, seed=42, secondary_weight=0.5, batch_size=1024, weight_decay=1e-2,
           epochs=2 if SMOKE_TEST else 30, ridge=1.0, chunk=8192)
ENCODERS = dict(
    birdmae=dict(dim=768, emb="birdmae_embeddings.npy", windows="birdmae_windows.csv", target="birdmae_soundscape_embeddings.npy",
               hidden=512, dropout=0.2, max_lr=3e-3, expected_first=0.9749))
# name -> map applied to the source windows; soundscape windows are never transformed
VARIANTS = (("distill", "control"), ("distill_coral", "coral_std"))
SIGMA_FLOOR = 1e-6                                                # guard only, for a constant dimension
SOUNDSCAPE_WEIGHT = 0.5                                           # literal loss weight per soundscape window
TEACHER_POWER, TEACHER_FLOOR = 1.3, 0.01
N_SOUNDSCAPE_WINDOWS = 12000
BUDGET_S = float(os.environ.get("M3_BUDGET_S", 30 * 60))          # env override: local guard test only
SAFETY = 1.25

def find_dir(marker, env):
    if os.environ.get(env):
        return Path(os.environ[env])
    hits = glob.glob(f"/kaggle/input/**/{marker}", recursive=True)
    if not hits:
        raise FileNotFoundError(f"{marker} not found under /kaggle/input")
    return Path(hits[0]).parent

DATA = find_dir("birdclef-2025/sample_submission.csv", "M3_DATA_DIR")
MELS = find_dir("train_index.csv", "M3_MELS_DIR")
SOUND = find_dir("soundscape_embed_meta.json", "M3_SOUNDSCAPE_DIR")
PERCH_DIR = find_dir("perch_head.npz", "M3_PERCH_DIR")            # the original A2 head, exact name
BIRDMAE_DIR = find_dir("birdmae_embeddings.npy", "M3_BIRDMAE_DIR")
DIRS = dict(birdmae=BIRDMAE_DIR)
print("Data:", DATA, "| M1 index:", MELS, "| soundscapes:", SOUND, "| perch:", PERCH_DIR, "| birdmae:", BIRDMAE_DIR)

## Split and labels (same table, call and seed as the original notebooks)

In [ ]:
index = pd.read_csv(MELS / "train_index.csv")      # same table, in the same order, that M1 split
species = list(pd.read_csv(DATA / "sample_submission.csv", nrows=0).columns[1:])
sp2idx = {s: i for i, s in enumerate(species)}
y_idx = index["primary_label"].astype(str).map(sp2idx)
assert y_idx.notna().all(), "primary_label not in submission columns"
y_idx = y_idx.astype(int).values

def parse_secondary(s):
    try:
        labs = ast.literal_eval(s) if isinstance(s, str) else []
    except (ValueError, SyntaxError):
        labs = []
    return [sp2idx[l] for l in labs if l in sp2idx]

sec_idx = [parse_secondary(s) for s in index["secondary_labels"]]

idx_tr, idx_va = train_test_split(
    np.arange(len(index)), test_size=CFG["val_frac"], stratify=y_idx, random_state=CFG["seed"])
assert not set(idx_tr) & set(idx_va)
print(len(index), "clips,", len(species), "species | train", len(idx_tr), "| val", len(idx_va), flush=True)

def macro_auc(y_true, y_pred):
    keep = y_true.sum(axis=0) > 0          # like the competition metric
    return roc_auc_score(y_true[:, keep], y_pred[:, keep], average="macro"), int(keep.sum())

def head_predict(head, emb):
    """Sigmoid probabilities from saved head arrays. Keep identical to the inference notebooks."""
    x = (emb - head["mean"]) / head["std"]
    n_layers = int(head["n_layers"])
    for i in range(n_layers):
        x = x @ head[f"W{i}"].T + head[f"b{i}"]
        if i < n_layers - 1:
            x = np.maximum(x, 0.0)       # ReLU
    return 1.0 / (1.0 + np.exp(-x))

## Transforms (float64, fitted on train-fold source windows and on the soundscape windows)

In [ ]:
def cov_of(X):
    """Covariance of the rows, float64, ddof 0 (matches numpy std)."""
    X = X - X.mean(axis=0)
    return (X.T @ X) / len(X)

def sym(C):
    return (C + C.T) / 2.0

def matrix_roots(C, name):
    """C^(1/2) and C^(-1/2) of a covariance that already includes +I, plus eigenvalue stats."""
    w, V = np.linalg.eigh(sym(C))
    if not w.min() >= CFG["ridge"] - 1e-6:
        raise RuntimeError(f"{name}: minimum eigenvalue {w.min()} below {CFG['ridge']}")
    stats = dict(min_eig=float(w.min()), max_eig=float(w.max()), cond=float(w.max() / w.min()))
    print(f"  eig {name:20s}: min {stats['min_eig']:.6g} | max {stats['max_eig']:.6g} | cond {stats['cond']:.6g}", flush=True)
    return sym((V * np.sqrt(w)) @ V.T), sym((V / np.sqrt(w)) @ V.T), stats

def floor_sigma(s, name):
    n = int((s < SIGMA_FLOOR).sum())
    if n:
        print(f"  WARNING {name}: {n} dimensions have std below {SIGMA_FLOOR}, floored", flush=True)
    return np.maximum(s, SIGMA_FLOOR)

def fit_transforms(S, T):
    """S = train-fold source rows, T = soundscape rows, both float64. Returns the fitted statistics and maps."""
    d = S.shape[1]; I = np.eye(d); r = CFG["ridge"]
    mu_s, mu_t = S.mean(axis=0), T.mean(axis=0)
    sig_s = floor_sigma(S.std(axis=0), "source"); sig_t = floor_sigma(T.std(axis=0), "target")
    cov_s, cov_t = cov_of(S), cov_of(T)
    _, inv_s, eig_rs = matrix_roots(sym(cov_s) + r * I, "raw source")
    root_t, _, eig_rt = matrix_roots(sym(cov_t) + r * I, "raw target")
    A_raw = inv_s @ root_t
    Csz = cov_s / np.outer(sig_s, sig_s); Ctz = cov_t / np.outer(sig_t, sig_t)
    _, inv_sz, eig_zs = matrix_roots(sym(Csz) + r * I, "standardised source")
    root_tz, _, eig_zt = matrix_roots(sym(Ctz) + r * I, "standardised target")
    A_std = inv_sz @ root_tz
    return dict(mu_s=mu_s, mu_t=mu_t, sig_s=sig_s, sig_t=sig_t, cov_s=cov_s, cov_t=cov_t, A_raw=A_raw, A_std=A_std,
                eig=dict(raw_source=eig_rs, raw_target=eig_rt, standardised_source=eig_zs, standardised_target=eig_zt))

def transform(fit, variant, x):
    """Rows of float64 features to float64 features. Applies the fitted map, never re-fits."""
    if variant == "meanstd":
        return (x - fit["mu_s"]) / fit["sig_s"] * fit["sig_t"] + fit["mu_t"]
    if variant == "coral_raw":
        return (x - fit["mu_s"]) @ fit["A_raw"] + fit["mu_t"]
    if variant == "coral_std":
        return (((x - fit["mu_s"]) / fit["sig_s"]) @ fit["A_std"]) * fit["sig_t"] + fit["mu_t"]
    raise ValueError(variant)

def apply_map(fit, variant, E):
    """E float32 to float32, in chunks so float64 is used only per block."""
    if variant == "control":
        return E
    out = np.empty(E.shape, np.float32)
    for i in range(0, len(E), CFG["chunk"]):
        out[i:i + CFG["chunk"]] = transform(fit, variant, E[i:i + CFG["chunk"]].astype(np.float64))
    if not np.isfinite(out).all():
        raise RuntimeError(f"{variant}: NaN or Inf after transform")
    return out

def distances(cov_a, mean_a, fit):
    """CORAL and mean distance to the target, in raw units and in target-standardised units (both sides scaled by sigma_t)."""
    d = cov_a.shape[0]; sig_t = fit["sig_t"]
    D = cov_a - fit["cov_t"]
    Dz = D / np.outer(sig_t, sig_t)
    return dict(coral_raw=float((D ** 2).sum() / (4 * d ** 2)), coral_z=float((Dz ** 2).sum() / (4 * d ** 2)),
                mean_l2_raw=float(np.linalg.norm(mean_a - fit["mu_t"])),
                mean_l2_z=float(np.linalg.norm((mean_a - fit["mu_t"]) / sig_t)))

def frob_change(A):
    I = np.eye(A.shape[0])
    return float(np.linalg.norm(A - I, "fro") / np.linalg.norm(I, "fro"))

## Teacher labels (original A2 head on the soundscape Perch embeddings)

In [ ]:
t_teacher = time.time()
windows_s = pd.read_csv(SOUND / "soundscape_windows.csv")
N_S = len(windows_s)
P_S = np.load(SOUND / "perch_soundscape_embeddings.npy").astype(np.float32)
bm_shape = np.load(SOUND / "birdmae_soundscape_embeddings.npy", mmap_mode="r").shape
# Both arrays follow the row order of soundscape_windows.csv (the embed notebook checked the two window lists are identical)
assert P_S.shape[0] == N_S, f"Perch soundscape rows {P_S.shape[0]} != {N_S} windows"
assert bm_shape[0] == N_S, f"Bird-MAE soundscape rows {bm_shape[0]} != {N_S} windows"
assert P_S.shape[1] == 1536 and bm_shape[1] == 768, (P_S.shape, bm_shape)
assert np.isfinite(P_S).all(), "non-finite Perch soundscape embeddings"
assert json.load(open(SOUND / "soundscape_embed_meta.json"))["windows"] == N_S
if not SMOKE_TEST:
    assert N_S == N_SOUNDSCAPE_WINDOWS, f"expected {N_SOUNDSCAPE_WINDOWS} soundscape windows, got {N_S}"

a2_species = json.load(open(PERCH_DIR / "species.json"))
a3_species = json.load(open(BIRDMAE_DIR / "species.json"))
assert a2_species == a3_species, "A2 and A3 species lists differ"
assert a2_species == species, "A2/A3 species list differs from sample_submission"
assert json.load(open(PERCH_DIR / "head_cfg.json"))["best_head"] == "mlp", "perch_head.npz is not the MLP head"

teacher_head = dict(np.load(PERCH_DIR / "perch_head.npz"))
assert int(teacher_head["n_layers"]) == 2 and teacher_head["W0"].shape[1] == 1536 and teacher_head["W1"].shape[0] == len(species)
teacher_p = head_predict(teacher_head, P_S).astype(np.float32)
soft = teacher_p ** TEACHER_POWER
soft[soft < TEACHER_FLOOR] = 0
assert soft.dtype == np.float32 and np.isfinite(soft).all() and soft.shape == (N_S, len(species))
del P_S, teacher_p

n_nonzero = (soft > 0).sum(axis=0); n_half = (soft >= 0.5).sum(axis=0); soft_sum = soft.sum(axis=0)
order = np.lexsort((-soft_sum, -n_nonzero))[:10]        # most windows with a non-zero label, ties by summed soft label
teacher_stats = dict(
    n_windows=N_S, power=TEACHER_POWER, floor=TEACHER_FLOOR, soft_label_stats_are_after_power_and_floor=True,
    fraction_nonzero=float((soft > 0).mean()), mean_max_per_window=float(soft.max(axis=1).mean()),
    windows_any_label_ge_0_5=int((soft.max(axis=1) >= 0.5).sum()),
    top10_species=[dict(species=species[i], windows_nonzero=int(n_nonzero[i]), windows_ge_0_5=int(n_half[i]), sum_soft=float(soft_sum[i])) for i in order])
print(f"Teacher: {N_S} windows | fraction non-zero {teacher_stats['fraction_nonzero']:.4f} | mean max per window {teacher_stats['mean_max_per_window']:.4f} | "
      f"windows with any label >= 0.5: {teacher_stats['windows_any_label_ge_0_5']}", flush=True)
for r in teacher_stats["top10_species"]:
    print("  ", r, flush=True)
teacher_s = time.time() - t_teacher

## Head training (recipe of the saved heads, plus the soundscape windows)

In [ ]:
def make_head(cfg):
    return nn.Sequential(nn.Linear(cfg["dim"], cfg["hidden"]), nn.ReLU(), nn.Dropout(cfg["dropout"]), nn.Linear(cfg["hidden"], len(species)))

def train_run(enc, cfg, d, variant, X, T32, soft):
    """One full run. The seed is reset here, so every variant shares initial weights, batches and dropout draws."""
    t_start = time.time()
    torch.manual_seed(CFG["seed"]); np.random.seed(CFG["seed"])
    mean, std = X[d.in_train].mean(axis=0), X[d.in_train].std(axis=0) + 1e-6
    # The standardiser comes from the train-fold source windows only (recipe unchanged); the soundscape windows use it too.
    n_src, n_t = len(X), len(T32)
    Z = torch.from_numpy(np.concatenate([(X - mean) / std, (T32 - mean) / std]))
    Y_all = torch.cat([d.Y_t, torch.from_numpy(soft)])
    W_all = torch.cat([d.W_t, torch.full((n_t,), SOUNDSCAPE_WEIGHT, dtype=torch.float32)])
    tr_rows = torch.from_numpy(np.concatenate([np.where(d.in_train)[0], np.arange(n_src, n_src + n_t)]))
    w_src = float(d.W_t.numpy()[d.in_train].sum()); w_sound = SOUNDSCAPE_WEIGHT * n_t
    share = w_sound / (w_src + w_sound)
    print(f"[{enc}/{variant}] rows: {len(tr_rows)} (source {int(d.in_train.sum())} + soundscape {n_t}) | loss weight: source {w_src:.1f}, soundscape {w_sound:.1f}, soundscape share {share:.4f}", flush=True)

    @torch.no_grad()
    def predict(model, rows):
        model.eval(); out = []
        for i in range(0, len(rows), 4096):
            out.append(torch.sigmoid(model(Z[torch.from_numpy(rows[i:i + 4096])])).numpy())
        return np.concatenate(out)

    def clip_max_auc(model):
        p = pd.DataFrame(predict(model, d.rows_all_va)).groupby(d.va_groups).max().loc[d.order_clips].values
        return macro_auc(d.Y_clip, p)[0]

    model = make_head(cfg)
    epochs = CFG["epochs"]
    steps = int(np.ceil(len(tr_rows) / CFG["batch_size"]))
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["max_lr"], weight_decay=CFG["weight_decay"])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=cfg["max_lr"], total_steps=epochs * steps, pct_start=0.1)
    best = dict(first=-1.0); name = f"{enc}/{variant}"
    for ep in range(1, epochs + 1):
        model.train(); t0 = time.time(); total = 0.0
        perm = tr_rows[torch.randperm(len(tr_rows))]
        for s in range(steps):
            b = perm[s * CFG["batch_size"]:(s + 1) * CFG["batch_size"]]
            bce = nn.functional.binary_cross_entropy_with_logits(model(Z[b]), Y_all[b], reduction="none")
            loss = (bce.mean(1) * W_all[b]).sum() / W_all[b].sum()
            opt.zero_grad(set_to_none=True)
            loss.backward(); opt.step(); sched.step()
            total += loss.item()
        first, n_cls = macro_auc(d.Y_va, predict(model, d.rows_va))
        flag = ""
        if first > best["first"]:
            best = dict(first=first, clip_max=clip_max_auc(model), best_epoch=ep, n_classes=n_cls,
                        state={k: v.clone() for k, v in model.state_dict().items()}); flag = "  <- best"
        print(f"[{name}] ep {ep:2d}/{epochs} | loss {total / steps:.4f} | first-5s {first:.4f} | {time.time() - t0:.1f}s{flag}", flush=True)
    model.load_state_dict(best["state"]); model.eval()

    # Save in the format of the original heads and check the numpy path used at inference
    linears = [m for m in model if isinstance(m, nn.Linear)]
    arrays = dict(mean=mean, std=std, n_layers=len(linears))
    for i, m in enumerate(linears):
        arrays[f"W{i}"] = m.weight.detach().numpy(); arrays[f"b{i}"] = m.bias.detach().numpy()
    path = OUT / f"{enc}_head_{variant}.npz"
    np.savez(path, **arrays)
    head = dict(np.load(path))
    assert np.allclose(head_predict(head, X[d.rows_va]), predict(model, d.rows_va), atol=1e-4), f"{name}: numpy head differs from torch head (val)"
    rng = np.random.default_rng(CFG["seed"])
    emb_rand = (mean + std * rng.standard_normal((256, cfg["dim"]))).astype(np.float32)
    with torch.no_grad():
        p_torch = torch.sigmoid(model(torch.from_numpy(((emb_rand - mean) / std).astype(np.float32)))).numpy()
    max_diff = float(np.abs(head_predict(head, emb_rand) - p_torch).max())
    assert max_diff < 1e-4, f"{name}: numpy head differs from torch head (random input): {max_diff}"
    return dict(val_auc_first5s=float(best["first"]), val_auc_clip_max=float(best["clip_max"]), best_epoch=int(best["best_epoch"]),
                val_classes=int(best["n_classes"]), soundscape_loss_weight_share=share, loss_weight_source=w_src, loss_weight_soundscape=w_sound,
                head_file=path.name, numpy_vs_torch_max_diff=max_diff, seconds=time.time() - t_start)

## Checks

In [ ]:
def check_budget(projected, limit=BUDGET_S):
    print(f"TIMING guard | projected notebook total {projected / 60:.1f} min (limit {limit / 60:.0f})", flush=True)
    if projected > limit:
        raise RuntimeError(f"Projected {projected / 60:.1f} min exceeds the {limit / 60:.0f} min limit, stopping")

def to_py(o):
    if isinstance(o, dict):
        return {k: to_py(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [to_py(v) for v in o]
    if isinstance(o, np.generic):
        return o.item()
    return o

## Prepare, fit, run both variants

In [ ]:
class Data:
    pass

def prepare(enc, cfg):
    t0 = time.time(); d = Data()
    cache = DIRS[enc]
    val_clips = pd.read_csv(cache / "val_clips.csv")
    if not np.array_equal(val_clips["clip"].values, idx_va):
        raise RuntimeError(f"{enc}: val_clips.csv does not match the recomputed split")
    if "filename" in index.columns and not (index["filename"].astype(str).values[idx_va] == val_clips["filename"].astype(str).values).all():
        raise RuntimeError(f"{enc}: val_clips.csv filenames do not match train_index.csv")
    E = np.load(cache / cfg["emb"]).astype(np.float32)
    windows = pd.read_csv(cache / cfg["windows"])
    if E.shape[1] != cfg["dim"] or len(E) != len(windows) or not np.isfinite(E).all():
        raise RuntimeError(f"{enc}: bad source embeddings {E.shape} vs {len(windows)} windows")
    T = np.load(SOUND / cfg["target"]).astype(np.float64)
    if T.shape[1] != cfg["dim"] or not np.isfinite(T).all():
        raise RuntimeError(f"{enc}: bad soundscape embeddings {T.shape}")
    d.E, d.T = E, T
    clip_of, win_of = windows["clip"].values, windows["win"].values
    d.in_train = np.isin(clip_of, idx_tr); in_val = np.isin(clip_of, idx_va)
    assert not (d.in_train & in_val).any() and (d.in_train | in_val).all()
    Y = np.zeros((len(windows), len(species)), np.float32)
    for row, c in enumerate(clip_of):
        Y[row, sec_idx[c]] = CFG["secondary_weight"]
        Y[row, y_idx[c]] = 1.0
    clip_windows = pd.Series(clip_of).map(pd.Series(clip_of).value_counts()).values
    d.W_t = torch.from_numpy((1.0 / clip_windows).astype(np.float32)); d.Y_t = torch.from_numpy(Y)
    d.rows_va = np.where(in_val & (win_of == 0))[0]
    d.Y_va = np.zeros((len(d.rows_va), len(species)), np.float32)
    d.Y_va[np.arange(len(d.rows_va)), y_idx[clip_of[d.rows_va]]] = 1.0
    d.rows_all_va = np.where(in_val)[0]
    d.va_groups = clip_of[d.rows_all_va]
    d.order_clips = np.unique(d.va_groups)
    d.Y_clip = np.zeros((len(d.order_clips), len(species)), np.float32)
    d.Y_clip[np.arange(len(d.order_clips)), y_idx[d.order_clips]] = 1.0
    print(f"[{enc}] windows: source {len(E)} | train {d.in_train.sum()} | val {in_val.sum()} | val first-5s {len(d.rows_va)} | target {len(T)}", flush=True)
    d.fit = fit_transforms(E[d.in_train].astype(np.float64), T)
    d.prep_s = time.time() - t0
    return d

results = dict(config=dict(soundscape_loss_weight=SOUNDSCAPE_WEIGHT, teacher="perch_head.npz on perch_soundscape_embeddings.npy", epochs=CFG["epochs"]),
               teacher=teacher_stats, variants={})
enc = "birdmae"; cfg = ENCODERS[enc]
d = prepare(enc, cfg)
assert len(d.T) == N_S, f"soundscape rows {len(d.T)} != {N_S}"
assert d.T.shape[0] == bm_shape[0]
T32 = d.T.astype(np.float32)                # target features stay untransformed in both variants
fit = d.fit
before = distances(fit["cov_s"], fit["mu_s"], fit)
results["prep_seconds"] = d.prep_s
for pos, (variant, map_name) in enumerate(VARIANTS):
    t0 = time.time()
    X = apply_map(fit, map_name, d.E)       # control returns the source rows unchanged
    S_new = X[d.in_train].astype(np.float64)
    after = distances(cov_of(S_new), S_new.mean(axis=0), fit)
    del S_new
    t_map = time.time() - t0
    print(f"[{enc}/{variant}] map {map_name}: distances before {before} | after {after}", flush=True)
    r = train_run(enc, cfg, d, variant, X, T32, soft)
    del X
    r.update(map=map_name, transform_seconds=t_map, distances_before=before, distances_after=after)
    results["variants"][variant] = r
    json.dump(to_py(results), open(OUT / "distill_results.json", "w"), indent=1)
    print(f"RESULT [{enc}/{variant}] first-5s {r['val_auc_first5s']:.4f} | clip-max {r['val_auc_clip_max']:.4f} | best ep {r['best_epoch']} | "
          f"head {r['seconds']:.0f}s | soundscape weight share {r['soundscape_loss_weight_share']:.4f}", flush=True)
    if pos == 0:
        check_budget(time.time() - T0 + SAFETY * (len(VARIANTS) - 1) * (r["seconds"] + t_map))

## Final outputs

In [ ]:
json.dump(species, open(OUT / "species.json", "w"))
# The inference notebooks read head_cfg.json next to the head; Bird-MAE's file hashes are carried over from the embed run.
embed_cfg = json.load(open(BIRDMAE_DIR / "head_cfg.json"))
json.dump(dict(best_head="mlp", hf_repo=embed_cfg["hf_repo"], hf_revision=embed_cfg["hf_revision"],
               birdmae_files_sha256=embed_cfg["birdmae_files_sha256"], transformers_pin=embed_cfg["transformers_pin"]),
          open(OUT / "head_cfg.json", "w"), indent=1)
results["total_seconds"] = time.time() - T0
results["teacher_seconds"] = teacher_s
json.dump(to_py(results), open(OUT / "distill_results.json", "w"), indent=1)
for name, r in results["variants"].items():
    print(f"{name:14s} | first-5s {r['val_auc_first5s']:.4f} | clip-max {r['val_auc_clip_max']:.4f} | best ep {r['best_epoch']} | {r['head_file']}")
parts = "/".join(f"{r['seconds'] + r['transform_seconds']:.0f}" for r in results["variants"].values())
print(f"TIMING | teacher {teacher_s:.0f}s | prep {results['prep_seconds']:.0f}s | runs {parts}s | notebook total {results['total_seconds'] / 60:.1f} min")